In [1]:
import os, sys, random, time, warnings, tarfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from pathlib import Path
from collections import Counter
import scipy.io
import timm
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.environ["PYTORCH_ALLOC_CONF"]            = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(42)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PLOT_DIR = '/kaggle/working/plots'
os.makedirs(PLOT_DIR, exist_ok=True)

RESULTS = {
    'exp1':        {'Aircraft':0.,'CUB':0.,'Flowers':0.,'Cars':0.,'Dogs':0.},
    'exp2_cub':    {100:{'Many':0.,'Medium':0.,'Few':0.,'All':0.},
                     50:{'All':0.}, 10:{'All':0.}},
    'exp2_flower': {100:{'Many':0.,'Medium':0.,'Few':0.,'All':0.},
                     50:{'All':0.}, 10:{'All':0.}},
    'exp3':        {1:0., 5:0., 10:0., 'all':0.},
    'exp4':        {'WB+Water':0.,'WB+Land':0.,'LB+Land':0.,'LB+Water':0.},
}
print(f"Device: {DEVICE}  |  Results store ✓")

Device: cuda  |  Results store ✓


In [2]:
# ══════════════════════════════════════════════════════════════════
# HDMix Core — label-preserving augmentation, zero pre-generation
# All ops on GPU inside @torch.no_grad()
# ══════════════════════════════════════════════════════════════════

MN = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
MS = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

def denorm(x): return (x*MS.to(x.device,x.dtype)+MN.to(x.device,x.dtype)).clamp(0,1)
def renorm(x): return (x.clamp(0,1)-MN.to(x.device,x.dtype))/MS.to(x.device,x.dtype)

def split_params(model):
    hk=('fc','head','classifier'); bb,hd=[],[]
    for n,p in model.named_parameters():
        (hd if any(k in n for k in hk) else bb).append(p)
    return bb,hd

@torch.no_grad()
def soft_saliency(x_norm, keep_ratio):
    x    = denorm(x_norm)
    gray = 0.2989*x[:,0:1]+0.5870*x[:,1:2]+0.1140*x[:,2:3]
    kx   = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]],
                          device=x.device,dtype=x.dtype).view(1,1,3,3)
    lap  = torch.tensor([[0,1,0],[1,-4,1],[0,1,0]],
                          device=x.device,dtype=x.dtype).view(1,1,3,3)
    sal  = (0.55*torch.sqrt(F.conv2d(gray,kx,padding=1)**2+
                            F.conv2d(gray,kx.transpose(-1,-2),padding=1)**2+1e-6)
            +0.45*F.conv2d(gray,lap,padding=1).abs())
    B,_,H,W = sal.shape
    yy,xx = torch.meshgrid(torch.linspace(-1,1,H,device=x.device),
                            torch.linspace(-1,1,W,device=x.device),indexing='ij')
    sal = sal*(0.55+0.45*torch.exp(-(xx**2+yy**2)/0.70).view(1,1,H,W).to(x.dtype))
    sal = sal/(sal.flatten(1).amax(1).view(B,1,1,1)+1e-6)
    q   = torch.quantile(sal.flatten(1),1.0-keep_ratio,dim=1).view(B,1,1,1)
    alpha = torch.sigmoid(12.0*(sal-q))
    alpha = F.max_pool2d(alpha,5,stride=1,padding=2)
    return F.avg_pool2d(alpha,3,stride=1,padding=1).clamp(0,1)

@torch.no_grad()
def hdmix(imgs, labels, model, epoch, total_epochs, cfg):
    """cfg passed explicitly — no global default. Skips during warmup."""
    if imgs.size(0)<2 or epoch<=cfg['warmup_epochs']:
        return imgs, labels
    alpha = soft_saliency(imgs, cfg['keep_ratio'])
    small = F.interpolate(alpha,(28,28),mode='bilinear',align_corners=False).flatten(1)
    bg    = 1.0-small
    scores= torch.mm(bg,bg.t()); scores.fill_diagonal_(-1e9)
    K     = min(cfg['K'],imgs.size(0)-1)
    idx   = scores.topk(K,dim=1).indices[:,epoch%K]
    src=denorm(imgs); tgt=denorm(imgs[idx])
    flat  = alpha.flatten(1).clamp(1e-6,1-1e-6)
    H_n   = (-(flat*flat.log()+(1-flat)*(1-flat).log()).mean(1)/0.6931).clamp(0,1)
    prog  = min(1.0,(epoch-cfg['warmup_epochs'])/max(1,total_epochs-cfg['warmup_epochs']))
    s_c   = cfg['s_min']+(cfg['s_max']-cfg['s_min'])*prog
    s     = (s_c-0.15*H_n).clamp(cfg['s_min'],s_c).view(-1,1,1,1)
    tgt_sm= F.avg_pool2d(tgt,5,stride=1,padding=2)
    bg_out= ((1-s)*tgt+s*tgt_sm+torch.randn_like(tgt)*(cfg['noise_scale']*s)).clamp(0,1)
    mixed = renorm(alpha*src+(1-alpha)*bg_out)
    if epoch>=cfg['filter_epoch']:
        model.eval()
        with torch.no_grad(), autocast():
            prob=torch.softmax(model(mixed),dim=1)
        conf,pred=prob.max(1)
        keep=((pred==labels)&(conf>=cfg['min_conf'])).float().view(-1,1,1,1)
        mixed=keep*mixed+(1-keep)*imgs
        model.train()
    return mixed, labels

def mixing_prob(epoch, cfg):
    return 0.0 if epoch<=cfg['warmup_epochs'] else cfg['p_flat']

# ── Per-dataset configs
DATASET_CFG = {
    # Exp1 — FGVC
    'aircraft': dict(warmup_epochs=60, p_flat=0.15, keep_ratio=0.50,
                     s_min=0.20, s_max=0.45, filter_epoch=60,
                     min_conf=0.75, K=2, noise_scale=0.008),
    'cub':      dict(warmup_epochs=8,  p_flat=0.35, keep_ratio=0.32,
                     s_min=0.30, s_max=0.80, filter_epoch=8,
                     min_conf=0.60, K=3, noise_scale=0.015),
    'flowers':  dict(warmup_epochs=8,  p_flat=0.40, keep_ratio=0.28,
                     s_min=0.30, s_max=0.85, filter_epoch=8,
                     min_conf=0.55, K=3, noise_scale=0.018),
    'cars':     dict(warmup_epochs=50, p_flat=0.20, keep_ratio=0.45,
                     s_min=0.25, s_max=0.55, filter_epoch=50,
                     min_conf=0.70, K=2, noise_scale=0.010),
    'dogs':     dict(warmup_epochs=10, p_flat=0.30, keep_ratio=0.35,
                     s_min=0.30, s_max=0.75, filter_epoch=10,
                     min_conf=0.60, K=3, noise_scale=0.015),
    # Exp2 — Long-tail (more aggressive, class-balanced loss handles tail)
    'cub_lt':   dict(warmup_epochs=5,  p_flat=0.45, keep_ratio=0.32,
                     s_min=0.30, s_max=0.80, filter_epoch=5,
                     min_conf=0.45, K=3, noise_scale=0.015),
    'flower_lt':dict(warmup_epochs=5,  p_flat=0.50, keep_ratio=0.28,
                     s_min=0.30, s_max=0.85, filter_epoch=5,
                     min_conf=0.45, K=3, noise_scale=0.018),
    # Exp3 — Few-shot (conservative — protect scarce samples)
    'fs_1':  dict(warmup_epochs=3, p_flat=0.12, keep_ratio=0.55,
                  s_min=0.15, s_max=0.38, filter_epoch=3,
                  min_conf=0.82, K=2, noise_scale=0.005),
    'fs_5':  dict(warmup_epochs=5, p_flat=0.22, keep_ratio=0.45,
                  s_min=0.20, s_max=0.52, filter_epoch=5,
                  min_conf=0.72, K=2, noise_scale=0.008),
    'fs_10': dict(warmup_epochs=5, p_flat=0.28, keep_ratio=0.40,
                  s_min=0.25, s_max=0.62, filter_epoch=5,
                  min_conf=0.65, K=3, noise_scale=0.010),
    'fs_all':dict(warmup_epochs=8, p_flat=0.35, keep_ratio=0.35,
                  s_min=0.30, s_max=0.75, filter_epoch=8,
                  min_conf=0.60, K=3, noise_scale=0.013),
    # Exp4 — Waterbird robustness
    'waterbird':dict(warmup_epochs=6, p_flat=0.38, keep_ratio=0.42,
                     s_min=0.25, s_max=0.72, filter_epoch=6,
                     min_conf=0.58, K=3, noise_scale=0.012),
}
print("HDMix core ✓")

HDMix core ✓


In [3]:
# ══════════════════════════════════════════════════════════════════
# Dataset loaders
# ══════════════════════════════════════════════════════════════════

PATHS = {
    'aircraft': '/kaggle/input/datasets/dedon002/this-data/fgvc-aircraft-2013b/fgvc-aircraft-2013b/data',
    'cub':      '/kaggle/input/datasets/dedon002/this-data/CUB_200_201.tgz',
    'flowers':  ('/kaggle/input/datasets/dedon002/this-data/102flowers.tgz',
                 '/kaggle/input/datasets/dedon002/this-data/imagelabels.mat'),
    'dogs':     '/kaggle/input/datasets/jessicali9530/stanford-dogs-dataset',
    'cars':     '/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder',
}

class ImgDataset(Dataset):
    def __init__(self,imgs,labels,tf):
        self.imgs=[Path(p) for p in imgs]; self.labels=labels; self.tf=tf
    def __len__(self): return len(self.imgs)
    def __getitem__(self,i):
        return self.tf(Image.open(self.imgs[i]).convert('RGB')), self.labels[i]

def split_cls(imgs,lbls,frac=0.15,seed=42):
    rng=random.Random(seed); g={}
    for img,lbl in zip(imgs,lbls): g.setdefault(lbl,[]).append(img)
    ti,tl,vi,vl=[],[],[],[]
    for lbl,items in g.items():
        rng.shuffle(items); n=max(1,int(len(items)*frac))
        vi+=items[:n]; vl+=[lbl]*n
        ti+=items[n:]; tl+=[lbl]*(len(items)-n)
    return ti,tl,vi,vl

def _extract_cub():
    dest=Path('/tmp/cub_extract')
    if not any(dest.rglob('*.jpg')):
        dest.mkdir(parents=True,exist_ok=True)
        with tarfile.open(PATHS['cub'],'r:*') as t: t.extractall(dest)
    root=list(dest.rglob('images.txt'))[0].parent
    id2p,id2s,id2l={},{},{}
    for line in open(root/'images.txt'):
        i,p=line.strip().split(); id2p[i]=root/'images'/p
    for line in open(root/'train_test_split.txt'):
        i,s=line.strip().split(); id2s[i]=int(s)
    for line in open(root/'image_class_labels.txt'):
        i,c=line.strip().split(); id2l[i]=int(c)-1
    return id2p,id2s,id2l,root

def load_aircraft(use_trainval=True):
    root=Path(PATHS['aircraft'])
    def rd(sp):
        imgs,lbls=[],[]
        with open(root/f'images_variant_{sp}.txt') as f:
            for line in f:
                p=line.strip().split(maxsplit=1)
                if len(p)==2:
                    imgs.append(root/'images'/f'{p[0]}.jpg'); lbls.append(p[1].strip())
        return imgs,lbls
    ti,tl=rd('train'); vi,vl=rd('val'); ei,el=rd('test')
    cls=sorted(set(tl+vl+el)); c2i={c:i for i,c in enumerate(cls)}
    ti_=[str(x) for x in ti]; tl_=[c2i[l] for l in tl]
    vi_=[str(x) for x in vi]; vl_=[c2i[l] for l in vl]
    ei_=[str(x) for x in ei]; el_=[c2i[l] for l in el]
    if use_trainval:
        full_i=ti_+vi_; full_l=tl_+vl_
        rng=random.Random(42); c2idx={}
        for i,(img,lbl) in enumerate(zip(full_i,full_l)):
            c2idx.setdefault(lbl,[]).append(i)
        val_idx=set()
        for lbl,idxs in c2idx.items():
            rng.shuffle(idxs); val_idx.update(idxs[:1])
        tr_idx=[i for i in range(len(full_i)) if i not in val_idx]
        ti_=[full_i[i] for i in tr_idx]; tl_=[full_l[i] for i in tr_idx]
        vi_=[full_i[i] for i in sorted(val_idx)]
        vl_=[full_l[i] for i in sorted(val_idx)]
    return ti_,tl_,vi_,vl_,ei_,el_,len(cls)

def load_cub():
    id2p,id2s,id2l,_=_extract_cub()
    ids=sorted(id2p)
    tr=[i for i in ids if id2s[i]==1]; te=[i for i in ids if id2s[i]==0]
    ti,tl,vi,vl=split_cls([id2p[i] for i in tr],[id2l[i] for i in tr])
    return ([str(x) for x in ti],tl,[str(x) for x in vi],vl,
            [str(id2p[i]) for i in te],[id2l[i] for i in te],max(id2l.values())+1)

def load_flowers():
    dest=Path('/tmp/flowers_extract')
    if not any(dest.rglob('*.jpg')):
        dest.mkdir(parents=True,exist_ok=True)
        with tarfile.open(PATHS['flowers'][0],'r:*') as t: t.extractall(dest)
    jpgs=sorted(dest.rglob('*.jpg'))
    lbls=(scipy.io.loadmat(PATHS['flowers'][1])['labels'].flatten()-1).tolist()
    rng=random.Random(42); cls2f={}
    for f,l in zip(jpgs,lbls): cls2f.setdefault(int(l),[]).append(f)
    ti,tl,vi,vl,ei,el=[],[],[],[],[],[]
    for l,files in cls2f.items():
        rng.shuffle(files)
        ti+=files[:10]; tl+=[l]*10
        vi+=files[10:20]; vl+=[l]*10
        ei+=files[20:]; el+=[l]*len(files[20:])
    return ([str(x) for x in ti],tl,[str(x) for x in vi],vl,
            [str(x) for x in ei],el,102)

def load_dogs():
    root=Path(PATHS['dogs']); img_root=root/'images'/'Images'
    if not img_root.exists(): img_root=list(root.rglob('Images'))[0]
    cls_names=sorted([d.name for d in img_root.iterdir() if d.is_dir()])
    c2i={c:i for i,c in enumerate(cls_names)}
    all_imgs,all_lbls=[],[]
    for cd in sorted(img_root.iterdir()):
        if not cd.is_dir(): continue
        l=c2i[cd.name]
        for jpg in sorted(cd.glob('*.jpg')): all_imgs.append(str(jpg)); all_lbls.append(l)
    rng=random.Random(42); c2imgs={}
    for img,lbl in zip(all_imgs,all_lbls): c2imgs.setdefault(lbl,[]).append(img)
    ti,tl,vi,vl,ei,el=[],[],[],[],[],[]
    for lbl,items in c2imgs.items():
        rng.shuffle(items)
        nt=max(1,int(len(items)*0.20)); nv=max(1,int(len(items)*0.15))
        ei+=items[:nt]; el+=[lbl]*nt
        vi+=items[nt:nt+nv]; vl+=[lbl]*nv
        ti+=items[nt+nv:]; tl+=[lbl]*(len(items)-nt-nv)
    return ti,tl,vi,vl,ei,el,len(cls_names)

def load_cars():
    root=Path(PATHS['cars'])
    tr_dir=root/'car_data'/'car_data'/'train'
    te_dir=root/'car_data'/'car_data'/'test'
    cls=sorted([d.name for d in tr_dir.iterdir() if d.is_dir()]); c2i={c:i for i,c in enumerate(cls)}
    def scan(d):
        imgs,lbls=[],[]
        for cd in sorted(d.iterdir()):
            if not cd.is_dir(): continue
            l=c2i.get(cd.name,0)
            for jpg in cd.glob('*.jpg'): imgs.append(str(jpg)); lbls.append(l)
        return imgs,lbls
    ti_r,tl=scan(tr_dir); ei,el=scan(te_dir); ti,tl,vi,vl=split_cls(ti_r,tl)
    return ti,tl,vi,vl,ei,el,len(cls)

def load_cub_fewshot(shot):
    id2p,id2s,id2l,_=_extract_cub()
    ids=sorted(id2p)
    tr=[i for i in ids if id2s[i]==1]; te=[i for i in ids if id2s[i]==0]
    rng=random.Random(42); c2tr={}
    for i in tr: c2tr.setdefault(id2l[i],[]).append(i)
    ti,tl=[],[]
    for cls_idx,iids in c2tr.items():
        rng.shuffle(iids)
        keep=iids if shot=='all' else iids[:shot]
        for iid in keep: ti.append(str(id2p[iid])); tl.append(cls_idx)
    ti,tl,vi,vl=split_cls(ti,tl,frac=0.15)
    ei=[str(id2p[i]) for i in te]; el=[id2l[i] for i in te]
    return ti,tl,vi,vl,ei,el,max(id2l.values())+1

def load_waterbird():
    id2p,id2s,id2l,root=_extract_cub()
    cls_names={}
    for line in open(root/'classes.txt'):
        c,n=line.strip().split(maxsplit=1); cls_names[int(c)-1]=n.lower()
    water_kw=['albatross','tern','gull','pelican','cormorant','gannet',
              'frigatebird','jaeger','petrel','booby','shearwater',
              'duck','goose','heron','egret','kingfisher','grebe',
              'loon','merganser','crane']
    wb_cls={i for i,n in cls_names.items() if any(k in n for k in water_kw)}
    def is_water_bg(p):
        n=Path(p).parent.name.lower()
        return any(k in n for k in ['water','ocean','lake','river','sea','beach','coast','pond'])
    ids=sorted(id2p)
    tr=[i for i in ids if id2s[i]==1]; te=[i for i in ids if id2s[i]==0]
    ti_r=[str(id2p[i]) for i in tr]
    tl_r=[1 if id2l[i] in wb_cls else 0 for i in tr]
    ti,tl,vi,vl=split_cls(ti_r,tl_r)
    te_imgs=[str(id2p[i]) for i in te]
    te_lbls=[1 if id2l[i] in wb_cls else 0 for i in te]
    groups=[(1 if id2l[i] in wb_cls else 0, 1 if is_water_bg(str(id2p[i])) else 0) for i in te]
    return ti,tl,vi,vl,te_imgs,te_lbls,groups,2

def make_longtail(imgs, lbls, imbalance_factor, seed=42):
    """
    CUB-LT/Flower-LT correct convention.
    Since CUB's natural per-class count (~30) never exceeds the universal
    Many>100/Medium 20-100/Few<20 thresholds from CIFAR-LT/ImageNet-LT,
    those absolute thresholds cannot work here — Many would always be empty.
    Instead, classes are ranked by their ORIGINAL (pre-imbalance) frequency
    and split into thirds: top third = Many (head), middle third = Medium,
    bottom third = Few (tail). This is the standard small-dataset long-tail
    convention and is what makes the paper's CUB-LT Many=79%+ numbers possible.
    """
    rng = random.Random(seed)
    c2imgs = {}
    for img, lbl in zip(imgs, lbls):
        c2imgs.setdefault(lbl, []).append(img)

    # Rank classes by ORIGINAL frequency (head -> tail)
    ranked = sorted(c2imgs.keys(), key=lambda k: -len(c2imgs[k]))
    n_cls  = len(ranked)
    max_n  = len(c2imgs[ranked[0]])
    min_n  = max(1, max_n // imbalance_factor)

    new_imgs, new_lbls = [], []
    for rank, cls_idx in enumerate(ranked):
        items = c2imgs[cls_idx]
        mu = max_n * (min_n / max_n) ** (rank / max(n_cls - 1, 1))
        n  = max(1, min(int(round(mu)), len(items)))
        rng.shuffle(items)
        new_imgs += items[:n]
        new_lbls += [cls_idx] * n

    # Bucket by rank-third, NOT by absolute post-imbalance count
    third      = n_cls // 3
    many_cls   = set(ranked[:third])
    medium_cls = set(ranked[third:2*third])
    few_cls    = set(ranked[2*third:])

    return new_imgs, new_lbls, many_cls, medium_cls, few_cls

print("Dataset loaders ✓")

Dataset loaders ✓


In [4]:
# ══════════════════════════════════════════════════════════════════
# Transforms
# ══════════════════════════════════════════════════════════════════

def _norm(): return T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

def train_tf(sz):
    return T.Compose([
        T.RandomResizedCrop(sz,scale=(0.40,1.0)),
        T.RandomHorizontalFlip(0.5),
        T.RandomVerticalFlip(0.05),
        T.RandAugment(num_ops=2,magnitude=10),
        T.ColorJitter(0.3,0.3,0.3,0.1),
        T.ToTensor(),_norm(),
        T.RandomErasing(p=0.20,scale=(0.02,0.12)),
    ])

def test_tf(sz):
    return T.Compose([T.Resize((sz,sz)),T.ToTensor(),_norm()])

def tta_tfs(sz):
    norm=_norm(); tfs=[]
    for scale in [1.0,1.1,1.15]:
        s=int(sz*scale)
        tfs+=[
            T.Compose([T.Resize((s,s)),T.CenterCrop(sz),T.ToTensor(),norm]),
            T.Compose([T.Resize((s,s)),T.CenterCrop(sz),
                       T.RandomHorizontalFlip(p=1.0),T.ToTensor(),norm]),
        ]
    tfs+=[T.Compose([T.Resize((sz,sz)),T.ToTensor(),norm]),
          T.Compose([T.Resize((sz,sz)),T.RandomHorizontalFlip(p=1.0),T.ToTensor(),norm]),
          T.Compose([T.Resize((int(sz*1.2),int(sz*1.2))),
                     T.FiveCrop(sz),T.Lambda(lambda c:c[0]),T.ToTensor(),norm])]
    return tfs

def make_loaders(ti,tl,vi,vl,ei,el,batch,sz):
    kw=dict(num_workers=2,pin_memory=True)
    return (DataLoader(ImgDataset(ti,tl,train_tf(sz)),batch,shuffle=True,drop_last=True,**kw),
            DataLoader(ImgDataset(vi,vl,test_tf(sz)), batch,shuffle=False,**kw),
            DataLoader(ImgDataset(ei,el,test_tf(sz)), batch,shuffle=False,**kw))

# ══════════════════════════════════════════════════════════════════
# Training utilities
# ══════════════════════════════════════════════════════════════════

def make_model(backbone, nc):
    if backbone=='resnet50':
        m=timm.create_model('resnet50.a1_in1k',pretrained=True,num_classes=nc,drop_rate=0.0)
    elif 'vit' in backbone:
        m=timm.create_model('vit_base_patch16_384.orig_in21k_ft_in1k',
                              pretrained=True,num_classes=nc,drop_rate=0.0)
    else:
        m=timm.create_model(backbone,pretrained=True,num_classes=nc,drop_rate=0.0)
    return m.to(DEVICE)

def make_optimizer(model, lr_bb=1e-4, lr_head=1e-3, wd=1e-4):
    bb,hd=split_params(model)
    return torch.optim.AdamW([{'params':bb,'lr':lr_bb},{'params':hd,'lr':lr_head}],
                              betas=(0.9,0.999),eps=1e-8,weight_decay=wd)

def make_scheduler(optimizer, epochs):
    def fn(ep):
        if ep<5: return (ep+1)/5
        t=(ep-5)/max(1,epochs-5)
        return max(1e-6,0.5*(1+np.cos(np.pi*t)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer,fn)

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); correct,total=0,0
    for imgs,labels in loader:
        imgs=imgs.to(DEVICE,non_blocking=True); labels=labels.to(DEVICE,non_blocking=True)
        with autocast(): out=model(imgs)
        correct+=(out.argmax(1)==labels).sum().item(); total+=imgs.size(0)
    return correct/total

@torch.no_grad()
def evaluate_tta(model,ei,el,sz,batch,save_dir):
    ckpt=torch.load(f'{save_dir}/best.pth',map_location=DEVICE)
    model.load_state_dict(ckpt['model']); model.eval()
    all_probs=[]
    for tf in tta_tfs(sz):
        ds=ImgDataset(ei,el,tf)
        ld=DataLoader(ds,batch,shuffle=False,num_workers=2,pin_memory=True)
        probs=[]
        for imgs,_ in ld:
            with autocast(): probs.append(torch.softmax(model(imgs.to(DEVICE)),1).cpu())
        all_probs.append(torch.cat(probs))
    preds=torch.stack(all_probs).mean(0).argmax(1)
    return (preds==torch.tensor(el)).float().mean().item()

def run_training(model,train_loader,val_loader,optimizer,scheduler,scaler,
                 epochs,cfg,save_dir,class_weight=None,label_smooth=0.05):
    best_val=0.
    for epoch in range(1,epochs+1):
        t0=time.time(); model.train(); loss_sum,correct,total=0.,0,0
        p_mix=mixing_prob(epoch,cfg)
        for imgs,labels in train_loader:
            imgs=imgs.to(DEVICE,non_blocking=True); labels=labels.to(DEVICE,non_blocking=True)
            optimizer.zero_grad()
            if p_mix>0 and random.random()<p_mix:
                imgs,labels=hdmix(imgs,labels,model,epoch,epochs,cfg)
            with autocast():
                out=model(imgs)
                loss=F.cross_entropy(out,labels,weight=class_weight,label_smoothing=label_smooth)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            scaler.step(optimizer); scaler.update()
            loss_sum+=loss.item()*imgs.size(0)
            correct+=(out.argmax(1)==labels).sum().item(); total+=imgs.size(0)
        va=evaluate(model,val_loader); scheduler.step()
        if va>best_val:
            best_val=va; torch.save({'model':model.state_dict()},f'{save_dir}/best.pth'); marker=' ← best'
        else: marker=''
        phase='CLEAN' if p_mix==0 else f'MIX p={p_mix:.2f}'
        print(f"Ep {epoch:3d}/{epochs} [{phase:<12}] loss={loss_sum/total:.4f} "
              f"tr={correct/total*100:.2f}% val={va*100:.2f}% | {time.time()-t0:.0f}s{marker}")
    return best_val

print("Transforms + training utils ✓")

Transforms + training utils ✓


## EXPERIMENT 2 — Long-Tail Classification
Set `DATASET` → `'cub'` | `'flowers'` and `IMBALANCE` → `100` | `50` | `10`.

In [11]:
DATASET   = 'cub'   # 'cub' | 'flowers'
IMBALANCE = 50      # 100 | 50 | 10
BACKBONE  = 'resnet50'
EPOCHS    = 100; BATCH=32; IMAGE_SIZE=448
SAVE_DIR  = f'/kaggle/working/exp2_{DATASET}_IF{IMBALANCE}'; os.makedirs(SAVE_DIR,exist_ok=True)
CFG_LT    = DATASET_CFG['cub_lt' if DATASET=='cub' else 'flower_lt']
print(f"Exp2 | {DATASET.upper()} IF={IMBALANCE} | warmup={CFG_LT['warmup_epochs']}")

Exp2 | CUB IF=50 | warmup=5


In [12]:
if DATASET=='cub':     _ti,_tl,_vi,_vl,ei,el,NC = load_cub()
else:                  _ti,_tl,_vi,_vl,ei,el,NC = load_flowers()

ti_lt, tl_lt, many_cls, medium_cls, few_cls = make_longtail(_ti, _tl, IMBALANCE)

print(f"Original train:{len(_ti)} → LT train:{len(ti_lt)}")
print(f"Many:{len(many_cls)} Medium:{len(medium_cls)} Few:{len(few_cls)} classes")
print(f"LT count range: {min(Counter(tl_lt).values())} – {max(Counter(tl_lt).values())}")

cnt_lt = Counter(tl_lt)
freq   = torch.tensor([cnt_lt.get(i,1) for i in range(NC)], dtype=torch.float)
weight = (1.0/freq); weight=(weight/weight.sum()*NC).to(DEVICE)

kw=dict(num_workers=2,pin_memory=True)
train_loader=DataLoader(ImgDataset(ti_lt,tl_lt,train_tf(IMAGE_SIZE)),BATCH,shuffle=True,drop_last=True,**kw)
val_loader  =DataLoader(ImgDataset(_vi,_vl,test_tf(IMAGE_SIZE)),BATCH,shuffle=False,**kw)
test_loader =DataLoader(ImgDataset(ei,el,test_tf(IMAGE_SIZE)),BATCH,shuffle=False,**kw)

Original train:5194 → LT train:1538
Many:66 Medium:66 Few:68 classes
LT count range: 1 – 26


In [13]:
model=make_model(BACKBONE,NC)
optimizer=make_optimizer(model,lr_bb=1e-4,lr_head=1e-3)
scheduler=make_scheduler(optimizer,EPOCHS)
scaler=GradScaler()
run_training(model,train_loader,val_loader,optimizer,scheduler,scaler,
             EPOCHS,CFG_LT,SAVE_DIR,class_weight=weight,label_smooth=0.05)

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Ep   1/100 [CLEAN       ] loss=5.7598 tr=0.78% val=1.75% | 40s ← best
Ep   2/100 [CLEAN       ] loss=5.6831 tr=1.95% val=5.75% | 38s ← best
Ep   3/100 [CLEAN       ] loss=5.5748 tr=4.30% val=9.25% | 38s ← best
Ep   4/100 [CLEAN       ] loss=5.3426 tr=13.15% val=11.75% | 36s ← best
Ep   5/100 [CLEAN       ] loss=4.7356 tr=23.70% val=19.12% | 35s ← best
Ep   6/100 [MIX p=0.45  ] loss=3.8683 tr=37.50% val=24.50% | 39s ← best
Ep   7/100 [MIX p=0.45  ] loss=3.0401 tr=50.33% val=31.13% | 38s ← best
Ep   8/100 [MIX p=0.45  ] loss=2.5390 tr=62.63% val=34.25% | 38s ← best
Ep   9/100 [MIX p=0.45  ] loss=2.2536 tr=68.36% val=37.88% | 39s ← best
Ep  10/100 [MIX p=0.45  ] loss=2.0438 tr=74.74% val=38.62% | 37s ← best
Ep  11/100 [MIX p=0.45  ] loss=1.9005 tr=77.73% val=41.75% | 39s ← best
Ep  12/100 [MIX p=0.45  ] loss=1.7846 tr=82.94% val=42.38% | 40s ← best
Ep  13/100 [MIX p=0.45  ] loss=1.6857 tr=84.11% val=42.50% | 39s ← best
Ep  14/100 [MIX p=0.45  ] loss=1.6336 tr=86.52% val=43.38% | 40s ← bes

0.515

In [14]:
@torch.no_grad()
def evaluate_splits():
    ckpt=torch.load(f'{SAVE_DIR}/best.pth',map_location=DEVICE)
    model.load_state_dict(ckpt['model']); model.eval()
    all_preds,all_labels=[],[]
    for imgs,labels in test_loader:
        with autocast(): out=model(imgs.to(DEVICE))
        all_preds.append(out.argmax(1).cpu()); all_labels.append(labels)
    preds=torch.cat(all_preds); targets=torch.cat(all_labels)
    acc_all=(preds==targets).float().mean().item()*100

    def acc(mask):
        if mask.sum()==0: return float('nan')
        return (preds[mask]==targets[mask]).float().mean().item()*100

    many_m  =torch.tensor([l.item() in many_cls   for l in targets])
    medium_m=torch.tensor([l.item() in medium_cls for l in targets])
    few_m   =torch.tensor([l.item() in few_cls    for l in targets])

    PAPER_LT={
        'cub':  {100:{'Many':84.34,'Medium':66.80,'Few':37.00,'All':49.48},
                  50:{'All':58.68}, 10:{'All':64.98}},
        'flowers':{100:{'Many':99.68,'Medium':96.30,'Few':79.98,'All':88.66},
                   50:{'All':94.03}, 10:{'All':96.89}},
    }
    print(f"\n{'='*65}")
    print(f"  Long-tail {DATASET.upper()} IF={IMBALANCE}")
    print(f"  (Many/Medium/Few defined on original balanced counts)")
    print(f"{'='*65}")
    print(f"  {'Split':<10} {'HDMix':>8}  {'SGD-Mix':>8}  {'Delta':>8}  {'N test':>7}")
    p=PAPER_LT[DATASET][IMBALANCE]; res={}
    for sname,mask,key in [('Many',many_m,'Many'),('Medium',medium_m,'Medium'),
                             ('Few',few_m,'Few'),('All',None,'All')]:
        ours=acc_all if mask is None else acc(mask)
        n_test=targets.size(0) if mask is None else mask.sum().item()
        pap=p.get(key); pstr=f'{pap:.2f}%' if pap else '—'
        dstr=f'{ours-pap:+.2f}%' if pap and not (ours!=ours) else '—'
        beat='✓' if (pap and not (ours!=ours) and ours>pap) else '✗'
        ostr=f'{ours:.2f}%' if not (ours!=ours) else 'N/A'
        print(f"  {sname:<10} {ostr:>8}  {pstr:>8}  {dstr:>8}  {n_test:>7}  {beat}")
        res[sname]=round(ours,2) if not (ours!=ours) else 0.
    print(f"{'='*65}")
    sk='exp2_cub' if DATASET=='cub' else 'exp2_flower'
    RESULTS[sk][IMBALANCE]=res
    print(f"Saved → RESULTS['{sk}'][{IMBALANCE}] = {res}")

evaluate_splits()


  Long-tail CUB IF=50
  (Many/Medium/Few defined on original balanced counts)
  Split         HDMix   SGD-Mix     Delta   N test
  Many         70.72%         —         —     1899  ✗
  Medium       57.75%         —         —     1910  ✗
  Few          19.70%         —         —     1985  ✗
  All          48.96%    58.68%    -9.72%     5794  ✗
Saved → RESULTS['exp2_cub'][50] = {'Many': 70.72, 'Medium': 57.75, 'Few': 19.7, 'All': 48.96}
